https://github.com/opencobra/optlang

https://blog.csdn.net/gitblog_00715/article/details/142122936

https://optlang.readthedocs.io/en/latest/

https://optlang.readthedocs.io/_/downloads/en/stable/pdf/

Optlang 是一个实现数学优化问题建模语言的 Python 软件包，用于在满足若干约束条件的前提下，通过一系列变量实现目标函数的最大化或最小化。Optlang 为多种优化工具提供了统一接口，使得不同求解器后端能够无缝切换。

与常用的通用代数建模系统（GAMS）不同，Optlang 具有基于原生 Python 代数语法的简洁直观接口，且是免费开源软件。该软件包利用符号数学库 SymPy，可轻松通过变量符号表达式构建目标函数和约束条件（参见示例）。科研人员因此能够运用领域知识衍生的数学表达式，通过 Optlang 构建优化问题。

当前支持的求解器包括：
• GLPK（线性规划/混合整数线性规划；通过 swiglpk 接口）
• CPLEX（线性规划/混合整数线性规划/二次规划）
• Gurobi（线性规划/混合整数线性规划/二次规划）
• inspyred（启发式优化；实验性支持）

计划支持的求解器包括：
• GAMS（线性规划/混合整数线性规划/二次规划/非线性规划；将包含对 neos-server.org 在线求解的支持）
• SOPLEX（精确线性规划）
• MOSEK（线性规划/混合整数线性规划/二次规划）

In [15]:
from optlang import Model, Variable, Constraint, Objective

 # All the (symbolic) variables are declared, with a name and optionally a lower and/or upper bound.
x1 = Variable('x1', lb=0)
x2 = Variable('x2', lb=0)
x3 = Variable('x3', lb=0)

# A constraint is constructed from an expression of variables and a lower and/or upper bound (lb and ub)
c1 = Constraint(x1 + x2 + x3, ub=100)
c2 = Constraint(10 * x1 + 4 * x2 + 5 * x3, ub=600)
c3 = Constraint(2 * x1 + 2 * x2 + 6 * x3, ub=300)

# An objective can be formulated
obj = Objective(10 * x1 + 6 * x2 + 4 * x3, direction='max')

# Variables, constraints and objective are combined in a Model object, which can subsequently be optimized.
model = Model(name='Simple model')
model.objective = obj
model.add([c1, c2, c3])
status = model.optimize()

In [3]:
print("status:", model.status)
print("objective value:", model.objective.value)
print("----------")
for var_name, var in model.variables.items():
    print(var_name, "=", var.primal)

status: optimal
objective value: 733.3333333333333
----------
x1 = 33.33333333333333
x2 = 66.66666666666667
x3 = 0.0


In [4]:
model.variables

In [5]:
model.variables.items()

<generator object Container.items.<locals>.<genexpr> at 0x7fec289f1ba0>

1.1 使用特定求解器  
若已安装多个求解器，可通过直接从相应求解器接口导入模块来指定使用某个求解器。例如：  
```python
from optlang.glpk_interface import Model, Variable, Constraint, Objective
```  

1.2 二次规划  
通过为目标函数创建包含二次项的表达式，可以用相同的方式生成二次规划（QP）问题。  
在以下示例中，目标函数可定义为：  
```python
obj = Objective(x1 ** 2 + x2 ** 2 - 10 * x1, direction="min")
```  
以此指定一个二次最小化问题。  

1.3 整数规划  
整数规划（或混合整数规划）问题可通过将一个或多个变量的类型设置为 `'integer'`（整数）或 `'binary'`（二元）来定义。若求解器支持整数规划，它将自动调用相关优化算法并返回整数解。

In [6]:
from optlang import Variable,Constraint,Objective,Model

# Define problem parameters
# Note this can be done using any of Python's data types. Here we have chosen dictionaries
supply = {"Seattle": 350, "San_Diego": 600}
demand = {"New_York": 325, "Chicago": 300, "Topeka": 275}

distances={ # Distances between locations in thousands of miles
    "Seattle": {"New_York": 2.5, "Chicago": 1.7, "Topeka": 1.8},
    "San_Diego": {"New_York": 2.5, "Chicago": 1.8, "Topeka": 1.4}
}

freight_cost=9 # Cost per case per thousand miles

In [7]:
# Define variables
variables = {}
for origin in supply:
    variables[origin] = {}
    for destination in demand:
        # Construct a variable with a name, bounds and type
        var = Variable(name="{}_to_{}".format(origin, destination), lb=0, type="integer")
        variables[origin][destination] = var

In [8]:
variables

{'Seattle': {'New_York': 0 <= Seattle_to_New_York,
  'Chicago': 0 <= Seattle_to_Chicago,
  'Topeka': 0 <= Seattle_to_Topeka},
 'San_Diego': {'New_York': 0 <= San_Diego_to_New_York,
  'Chicago': 0 <= San_Diego_to_Chicago,
  'Topeka': 0 <= San_Diego_to_Topeka}}

In [9]:
# Define constraints
constraints =[]
for origin in supply:
    const=Constraint(
        sum(variables[origin].values()),
        ub=supply[origin],
        name="{}_supply".format(origin)
        )
    constraints.append(const)

In [10]:
constraints

In [11]:
for destination in demand:
    const = Constraint(
        sum(row[destination] for row in variables.values()),
        lb=demand[destination],
        name="{}_demand".format(destination)
    )
    constraints.append(const)

In [12]:
const

In [13]:
# Define the objective
obj = Objective(
    sum(freight_cost * distances[ori][dest] * variables[ori][dest] for ori in supply
        for dest in demand),
    direction="min"
)

以下是这段代码所描述的数学问题的详细解释：

---

### **问题类型**
这是一个经典的 **运输问题（Transportation Problem）**，属于 **线性规划（Linear Programming）** 的范畴。  
**目标**：在满足供应和需求约束的前提下，找到总运输成本最小的货物分配方案。

---

### **问题场景**
假设你是一家物流公司的经理，需要从两个仓库（西雅图、圣地亚哥）向三个城市（纽约、芝加哥、托皮卡）运输货物。  
- **仓库供应量**：西雅图 350 箱，圣地亚哥 600 箱。
- **城市需求量**：纽约 325 箱，芝加哥 300 箱，托皮卡 275 箱。
- **运输成本**：每箱货物每千英里运费为 9 美元。
- **运输距离**：已知各仓库到城市的距离（单位为千英里）。

你需要决定从每个仓库到每个城市运输多少箱货物，使得 **总运输成本最低**。

---

### **数学建模**
#### **1. 决策变量**
定义变量 `x_ij`，表示从仓库 `i` 运输到城市 `j` 的货物数量（单位：箱）。  
例如：
- `x_Seattle_to_New_York`: 西雅图到纽约的运输量。
- `x_San_Diego_to_Topeka`: 圣地亚哥到托皮卡的运输量。

在代码中，变量通过以下方式定义：
```python
var = Variable(name="{}_to_{}".format(origin, destination), lb=0, type="integer")
```
- `lb=0`: 运输量不能为负数。
- `type="integer"`: 运输量必须是整数（实际场景中货物通常按整箱运输）。

---

#### **2. 约束条件**
- **供应约束**：每个仓库的运出量不能超过其供应量。  
  $$
  \sum_{j} x_{ij} \leq \text{Supply}_i \quad \forall i
  $$
  例如：  
  $$
  x_{\text{Seattle}\to\text{New York}} + x_{\text{Seattle}\to\text{Chicago}} + x_{\text{Seattle}\to\text{Topeka}} \leq 350
  $$

- **需求约束**：每个城市的运入量必须满足其需求量。  
  $$
  \sum_{i} x_{ij} \geq \text{Demand}_j \quad \forall j
  $$
  例如：  
  $$
  x_{\text{Seattle}\to\text{New York}} + x_{\text{San Diego}\to\text{New York}} \geq 325
  $$

在代码中，约束通过以下方式定义：
```python
# 供应约束
const = Constraint(sum(variables[origin].values()), ub=supply[origin], ...)
# 需求约束
const = Constraint(sum(row[destination] for row in variables.values()), lb=demand[destination], ...)
```

---

#### **3. 目标函数**
最小化总运输成本：  
$$
\text{Minimize} \quad \sum_{i,j} 9 \times \text{Distance}_{ij} \times x_{ij}
$$
其中：
- `9` 是每箱每千英里的运费。
- `Distance_{ij}` 是仓库 `i` 到城市 `j` 的距离（千英里）。
- `x_{ij}` 是运输量（箱）。

在代码中，目标函数通过以下方式定义：
```python
obj = Objective(
    sum(freight_cost * distances[ori][dest] * variables[ori][dest] for ori, dest in ...),
    direction="min"
)
```

---

### **示例计算**
假设从西雅图到纽约运输 100 箱，距离为 2.5 千英里：
$$
\text{成本} = 9 \, (\text{美元/箱/千英里}) \times 2.5 \, (\text{千英里}) \times 100 \, (\text{箱}) = 2250 \, \text{美元}
$$

---

### **数据验证**
- **总供应量**：350 + 600 = 950 箱。
- **总需求量**：325 + 300 + 275 = 900 箱。
- **供应 ≥ 需求**：950 ≥ 900，问题可行（有解）。

---

### **问题扩展**
- **不平衡运输问题**：若供应量 < 需求量，需引入虚拟仓库或调整需求约束为软约束。
- **多商品运输**：同时运输多种货物，需增加商品维度。
- **非线性成本**：若运费随运输量变化（如折扣），需使用非线性规划。

---

### **总结**
这段代码构建了一个 **运输优化模型**，通过线性规划求解最小化总运输成本的货物分配方案。核心步骤包括：
1. 定义变量（运输量）。
2. 添加约束（供应限制、需求满足）。
3. 定义目标函数（总运输成本最小化）。

实际应用场景包括物流调度、供应链管理、资源分配等。

In [14]:
# We can print the objective and constraints
print(obj)
print("")

for const in constraints:
    print(const)
print("")

# Pute verything together in a Model
model = Model()
model.add(constraints) # Variables are added implicitly
model.objective = obj
#Optimizeandprintthesolution
status = model.optimize()
print("Status:",status)
print("Objectivevalue:",model.objective.value)
print("")

for var in model.variables:
    print(var.name,":",var.primal)

Minimize
16.2*San_Diego_to_Chicago + 22.5*San_Diego_to_New_York + 12.6*San_Diego_to_Topeka + 15.3*Seattle_to_Chicago + 22.5*Seattle_to_New_York + 16.2*Seattle_to_Topeka

Seattle_supply: Seattle_to_Chicago + Seattle_to_New_York + Seattle_to_Topeka <= 350
San_Diego_supply: San_Diego_to_Chicago + San_Diego_to_New_York + San_Diego_to_Topeka <= 600
New_York_demand: 325 <= San_Diego_to_New_York + Seattle_to_New_York
Chicago_demand: 300 <= San_Diego_to_Chicago + Seattle_to_Chicago
Topeka_demand: 275 <= San_Diego_to_Topeka + Seattle_to_Topeka

Status: optimal
Objectivevalue: 15367.5

Seattle_to_New_York : 50.0
Seattle_to_Chicago : 300.0
Seattle_to_Topeka : 0.0
San_Diego_to_New_York : 275.0
San_Diego_to_Topeka : 275.0
San_Diego_to_Chicago : 0.0


3.1.1 求解器  
要解决优化问题，必须至少安装一个支持的求解器。使用 pip 安装 Optlang 时会自动安装 GLPK。若要使用其他求解器（如商业求解器），需手动安装。Optlang 通过可导入的 Python 模块与所有求解器交互。如果对应求解器的 Python 模块可成功导入，则该求解器接口将作为 Optlang 的子模块提供（例如 `optlang.glpk_interface`）。

当前支持的求解器所需的 Python 模块如下：  
• **GLPK**: `swiglpk`（通过 `pip install optlang` 自动安装）  
  GLPK 是一个开源的线性规划库。`swiglpk` 可通过二进制包或源码安装。从源码安装需预先安装 SWIG 和 GLPK。  

• **CPLEX**: `cplex`  
  CPLEX 是 IBM 开发的高效商用线性及二次混合整数规划求解器。学生和研究人员可申请学术许可证。  

• **Gurobi**: `gurobipy`  
  Gurobi 是高效商用线性及二次混合整数规划求解器。学生和研究人员可申请学术许可证。  

• **SciPy**: `scipy.optimize.linprog`  
  SciPy 的 `linprog` 函数是基于单纯形法的基本线性优化求解器，已包含在近期版本的 SciPy 中。  

导入 Optlang 后，可通过检查 `optlang.available_solvers` 确认系统是否已识别某求解器。

这张图是 **Optlang 框架中添加新求解器（名为 XYZ）接口的技术文档说明**，主要指导开发者如何通过继承现有类来扩展 Optlang 的功能，具体步骤可总结如下：

---

### **核心流程解析**
1. **创建接口文件**  
   • 需创建新文件 `xyz_interface.py`，并继承 Optlang 的 `interface.Model` 基类。  
   • 示例：  
     ```python
     class Model(interface.Model):
         def __init__(self, problem=None, *args, **kwargs):
             super(Model, self).__init__(*args, **kwargs)  # 初始化父类
     ```

2. **重写抽象方法**  
   • 需重写基类中定义的抽象方法（如 `_add_constraint()`），以适配新求解器的规则。  
   • 例如，若 XYZ 仅支持线性约束，需在 `_add_constraint` 中增加约束类型的验证逻辑：  
     ```python
     if not constraint.is_Linear:  # 检查约束是否线性
         raise ValueError("XYZ only supports linear constraints.")
     ```

3. **处理约束与变量**  
   • **步骤一**：调用父类的 `_add_constraint` 方法，将约束添加到用户级接口。  
   • **步骤二**：遍历约束中的所有变量，确保变量已添加到模型中（如未添加则需调用 `self._add_variable`）。  
   • **步骤三**：调用求解器 XYZ 的底层 API，将该约束添加到求解器实例（例如 `xyz_add_constraints` 和 `xyz_set_row_name`）。

---

### **关键设计思想**
• **兼容性**：通过继承和重写 Optlang 的接口类，确保新求解器 XYZ 遵循统一规范，最终用户在切换不同求解器（如 GLPK、Gurobi 或 XYZ）时，语法和流程完全一致。  
• **校验逻辑**：针对 XYZ 的特性（如仅支持线性）添加校验，避免不支持的约束类型传递到求解器底层导致的错误。  
• **错误处理与数据同步**：确保变量和约束在用户级接口与求解器底层数学模型中的状态同步（例如未声明的变量需自动添加到模型）。

---

### **示例代码的作用说明**
以下代码片段展示了如何为 XYZ 实现 `_add_constraint` 方法：  
```python
def _add_constraint(self, constraint):
    if not constraint.is_Linear:  # 校验约束类型
        raise ValueError("XYZ only supports linear constraints.")  
    super(Model, self)._add_constraint(constraint)  # 调用父类方法添加约束到用户接口
    
    # 添加约束中未声明的变量到模型
    for var in constraint.variables:
        if var.name not in self.variables:
            self._add_variable(var)
    
    # 调用 XYZ 底层 API，将约束关联到求解器数学模型
    xyz_add_constraints(self.problem, constraint)
    index = xyz_get_row_index(self.problem, constraint)
    xyz_set_row_name(self.problem, index, constraint.name)
```

---

### **总结**
这段文档提供了一套 **标准化流程**，帮助开发者为新求解器 XYZ 添加 Optlang 接口。通过继承基类并重写关键方法，既能复用 Optlang 的通用逻辑（如变量/约束管理），又可灵活实现 XYZ 的特殊需求（如线性约束校验）。最终用户可通过 `from optlang.xyz_interface import Model` 无缝使用 XYZ。